In [1]:
import numpy as np
import re

from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import CXGate,  PauliEvolutionGate, QAOAAnsatz
from qiskit.quantum_info import SparsePauliOp


from qiskit.transpiler import PassManager
from qiskit.transpiler.passes import InverseCancellation, CommutativeCancellation
from qopt_best_practices.transpilation.swap_cancellation_pass import SwapToFinalMapping

from qiskit_aer import AerSimulator
from qiskit_aer.backends.backendconfiguration import AerBackendConfiguration

from qiskit_qaoa.utils.transpiler_passes import ExtendedSwapStrategy, FindCommutingPauliEvolutionsMulti
from qiskit_qaoa.utils.commuting_gate_router_rzz import CommutingGateRouterRzz
from qiskit_qaoa.utils.commuting_gate_router import CommutingGateRouter
from qiskit_qaoa.hubo.graph_to_hubo_hamiltonian import graph_to_hubo_hamiltonian
from qiskit_qaoa.utils.gfa_utils import gfa_file_to_graph


In [2]:
def print_circuit_info(qc: QuantumCircuit, circuit_name):
    print(f'{circuit_name} has {qc.num_qubits} qubits')
    print(f'{circuit_name} has {qc.num_nonlocal_gates()} non-local gates and {qc.depth(lambda instr: len(instr.qubits) > 1)} non-local depth')
    print(f'{circuit_name} contains {list(qc.count_ops().keys())} gates.')

In [3]:
filepath = '/nfs/users/nfs_j/jc59/quantumwork/pangenome/data/test_N2_W2.gfa'
graph, n, V, T = gfa_file_to_graph(filepath, [1,1])
hamiltonian = graph_to_hubo_hamiltonian(graph, n, T, lamda=10, constraint_terms=1.0)
ess = ExtendedSwapStrategy.from_grid(n, T)
num_physical_qubits = ess._num_vertices
max_layers = 0

Keeping constraints at times: [0]


In [4]:
hamiltonian = SparsePauliOp.from_sparse_list(
    [
        x for x in hamiltonian.to_sparse_list() 
        if any([set(x[1]).__eq__(y) for y in [set((1,2,3)), set((1,2,3,4,5)), set((0,1,2,3,4,5)), set((0,1,3,5)), set((1,3,5))]])
    ],
    num_physical_qubits
)

In [5]:
method = 'unitary'
backend_options = dict(
    method=method,
    device='CPU',
    precision='single',
    basis_gates=["sx", "x", "rz", "rzz", "cz", "id", "cx"]
)

config = AerSimulator._DEFAULT_CONFIGURATION
config["n_qubits"] = num_physical_qubits
config = AerBackendConfiguration.from_dict(config)
backend = AerSimulator(configuration=config, coupling_map=ess._coupling_map, **backend_options)
backend.set_option("n_qubits", num_physical_qubits)
print(f'Num qubits in backend: {backend.configuration().to_dict()["n_qubits"]}')

Num qubits in backend: 6


In [6]:
default = QAOAAnsatz(hamiltonian, reps=1, initial_state=QuantumCircuit(num_physical_qubits), mixer_operator=QuantumCircuit(num_physical_qubits))
t_default = transpile(default, backend, optimization_level=3,basis_gates=["sx", "x", "rz", "rzz", "cz", "id", "cx"])
t_default.draw(fold=-1)

/nfs/users/nfs_j/jc59/quantumwork/pangenome/.venv/lib/python3.10/site-packages/qiskit/compiler/transpiler.py:269: UserWarning: Providing `coupling_map` and/or `basis_gates` along with `backend` is not recommended, as this will invalidate the backend's gate durations and error rates.
  pm = generate_preset_pass_manager(


┌───┐┌────────────────┐┌───┐                                                               ┌───┐┌────────────────┐┌───┐                    
q_0 -> 0 ─────────────────────────────────────────────────────────────────────────────────┤ X ├┤ Rz(0.625*γ[0]) ├┤ X ├───────────────────────────────────────────────────────────────┤ X ├┤ Rz(0.625*γ[0]) ├┤ X ├────────────────────
                                                                                          └─┬─┘└────────────────┘└─┬─┘                                                               └─┬─┘└────────────────┘└─┬─┘                    
q_5 -> 1 ───────────────────────────────────────────────────────■───────────────────────────┼──────────────────────┼─────────■───────────────────■─────────────────────────────────────┼──────────────────────┼───────────────────■──
              ┌───┐┌────────────────┐┌───┐                      │  ┌───┐┌────────────────┐  │                      │  ┌───┐  │                   │            ┌───┐┌────────────────┐  │                      │  ┌───┐            │  
q_1 -> 2 ─────┤ X ├┤ Rz(0.625*γ[0]) ├┤ X ├──────────────────────┼──┤ X ├┤ Rz(0.625*γ[0]) ├──■──────────────────────■──┤ X ├──┼───────────────────┼────────────┤ X ├┤ Rz(0.625*γ[0]) ├──■──────────────────────■──┤ X ├────────────┼──
              └─┬─┘└────────────────┘└─┬─┘     ┌───┐     ┌───┐┌─┴─┐└─┬─┘└────────────────┘                            └─┬─┘┌─┴─┐┌───┐     ┌───┐┌─┴─┐          └─┬─┘└────────────────┘                            └─┬─┘          ┌─┴─┐
q_4 -> 3 ───────┼──────────────────────┼───────┤ X ├──■──┤ X ├┤ X ├──■──────────────────────────────────────────────────■──┤ X ├┤ X ├──■──┤ X ├┤ X ├──■─────────┼──────────────────────────────────────────────────┼─────────■──┤ X ├
         ┌───┐  │                      │  ┌───┐└─┬─┘  │  └─┬─┘└───┘                                                        └───┘└─┬─┘  │  └─┬─┘└───┘  │  ┌───┐  │                                                  │  ┌───┐  │  └───┘
q_2 -> 4 ┤ X ├──■──────────────────────■──┤ X ├──┼────┼────┼──────────────────────────────────────────────────────────────────────┼────┼────┼─────────┼──┤ X ├──■──────────────────────────────────────────────────■──┤ X ├──┼───────
         └─┬─┘                            └─┬─┘  │  ┌─┴─┐  │                                                                      │  ┌─┴─┐  │       ┌─┴─┐└─┬─┘                                                        └─┬─┘┌─┴─┐     
q_3 -> 5 ──■────────────────────────────────■────■──┤ X ├──■──────────────────────────────────────────────────────────────────────■──┤ X ├──■───────┤ X ├──■────────────────────────────────────────────────────────────■──┤ X ├─────
                                                    └───┘                                                                            └───┘          └───┘                                                                  └───┘

In [7]:
print_circuit_info(t_default, 'Default QAOA')

Default QAOA has 6 qubits
Default QAOA has 26 non-local gates and 26 non-local depth
Default QAOA contains ['cx', 'rz'] gates.


In [26]:
pm = PassManager(
    [
        FindCommutingPauliEvolutionsMulti(), 
        CommutingGateRouterRzz(
            ess,
            max_layers=max_layers,
            perform_extra_swaps=True
        ),
        SwapToFinalMapping(),
        InverseCancellation(gates_to_cancel=[CXGate()]),
        CommutativeCancellation(basis_gates=["cx", "swap", "rz", "rzz"]),
        InverseCancellation(gates_to_cancel=[CXGate()]),
    ]
)

In [27]:
# gates = ['IIZIIZ', 'ZZIIII', 'ZZZIII', 'ZZZZII', 'ZZIZII']
# hamiltonian = SparsePauliOp(gates, range(1, len(gates) + 1))

In [28]:
# def choice_to_pauli(choice: list[int], size: int) -> str:
#     pauli = ['I'] * size
#     for x in choice:
#         pauli[size - x - 1] = 'Z'
#     return ''.join(pauli)

In [29]:
# rng = np.random.default_rng(seed=1)
# doubles = rng.choice(num_physical_qubits, (20, 2), replace = True)
# trips = rng.choice(num_physical_qubits, (20, 3), replace=True)
# quads = rng.choice(num_physical_qubits, (30, 4), replace=True)
# coeffs = rng.random(len(doubles) + len(trips) + len(quads))

# hamiltonian = SparsePauliOp(
#     [choice_to_pauli(c, num_physical_qubits) for c in doubles] + [choice_to_pauli(c, num_physical_qubits) for c in trips] + [choice_to_pauli(c, num_physical_qubits) for c in quads],
#     coeffs=coeffs
# )
# hamiltonian = hamiltonian.simplify()
# hamiltonian = hamiltonian.sort(weight=True)

In [30]:
qc = QuantumCircuit(num_physical_qubits)
qc.append(PauliEvolutionGate(hamiltonian), range(num_physical_qubits))
tqc = pm.run(qc)

print_circuit_info(tqc, 'Rzz circuit')

swaps = []
for instruction in tqc.data:
    if instruction.operation.name == 'swap':
        qubits_str = str(instruction.qubits)
        matches = re.findall('index=([0-9]+)', qubits_str)
        if len(matches) == 2:
            swaps.append((int(matches[0]), int(matches[1])))
        else:
            raise Exception('Did not find 2 swap indices')
for swap in swaps[::-1]:
    tqc.swap(*swap)

tqc.save_unitary()

Max layers needed to apply swap decompose: 0
{0: {0}, 1: {1}, 2: {2}, 3: {2, 3}, 4: {4}, 5: {5}}
{0: {0}, 1: {1}, 2: {2}, 3: {2, 3, 4, 5}, 4: {4}, 5: {4, 5}}
{0: {0}, 1: {0, 1}, 2: {2}, 3: {2, 3, 4, 5}, 4: {4}, 5: {4, 5}}
{0: {0}, 1: {0, 1}, 2: {2, 4}, 3: {3, 5}, 4: {4}, 5: {4, 5}}
{0: {0}, 1: {1}, 2: {2, 4}, 3: {3, 5}, 4: {4}, 5: {4, 5}}
Gates we cannot directly implement: 0
[]
Transpiling accumulator
Rzz circuit has 6 qubits
Rzz circuit has 17 non-local gates and 20 non-local depth
Rzz circuit contains ['cx', 'rzz', 'barrier'] gates.


In [31]:
tqc.draw(fold=-1)

(1, 2, 3)                        (1, 2, 3, 4, 5)                   (0, 1, 2, 3, 4, 5)                        (0, 1, 3, 5)                   (1, 3, 5)                      unitary 
q_0 -> 0 ──────────────────────░───────────────────────────────────░──────────■───────────────────────░──────────────────────────────────────░─────────■───────────────────░─────────────────────────────░────
                               ░                                   ░        ┌─┴─┐                     ░                                      ░       ┌─┴─┐                 ░                             ░    
q_1 -> 1 ──────■───────────────░────────────────■──────────────────░────────┤ X ├─■───────────────────░─────────────────────■────────────────░───────┤ X ├─■───────────────░─────────────────────────────░────
               │               ░                │                  ░        └───┘ │                   ░          ┌───┐      │                ░       └───┘ │               ░          ┌───┐              ░    
q_2 -> 2 ──■───┼───────────────░────────────────┼──────────────────░──────────────┼───────────────────░──────────┤ X ├──■───┼────────────────░─────────────┼───────────────░───────■──┤ X ├───────■──────░────
         ┌─┴─┐ │ZZ(0.625)      ░          ┌───┐ │ZZ(0.625)         ░              │ZZ(0.625)          ░          └─┬─┘┌─┴─┐ │ZZ(0.625)       ░             │ZZ(0.625)      ░     ┌─┴─┐└─┬─┘┌───┐┌─┴─┐    ░    
q_3 -> 3 ┤ X ├─■───────────────░──────────┤ X ├─■──────────────────░──────────────■───────────────────░────────────┼──┤ X ├─■────────────────░─────────────■───────────────░─────┤ X ├──┼──┤ X ├┤ X ├────░────
         └───┘                 ░          └─┬─┘                    ░                                  ░            │  └───┘                  ░                             ░     └───┘  │  └─┬─┘└───┘    ░    
q_4 -> 4 ──────────────────────░───────■────┼──────────────────────░──────────────────────────────────░────────────■─────────────────────────░─────────────────────────────░────────────■────┼────■──────░────
                               ░     ┌─┴─┐  │                      ░                                  ░                                      ░                             ░                 │  ┌─┴─┐    ░    
q_5 -> 5 ──────────────────────░─────┤ X ├──■──────────────────────░──────────────────────────────────░──────────────────────────────────────░─────────────────────────────░─────────────────■──┤ X ├────░────
                               ░     └───┘                         ░                                  ░                                      ░                             ░                    └───┘    ░

Rotation site (1,3)
Gates (1,2,3), (1,2,3,4,5), (0,1,2,3,4,5), (0,1,3,5), (1,3,5)

In [14]:
print_circuit_info(tqc, 'Rzz circuit')

Rzz circuit has 6 qubits
Rzz circuit has 17 non-local gates and 21 non-local depth
Rzz circuit contains ['cx', 'rzz', 'barrier', 'save_unitary'] gates.


In [15]:
res_rzz = backend.run(tqc).result()
data = res_rzz.results[0].data
unitary = np.asarray(data.unitary)

In [16]:
pm_rz = PassManager(
    [
        FindCommutingPauliEvolutionsMulti(), 
        CommutingGateRouter(
            ess,
            max_layers=max_layers,
            perform_extra_swaps=True
        ),
        SwapToFinalMapping(),
        InverseCancellation(gates_to_cancel=[CXGate()]),
        CommutativeCancellation(basis_gates=["cx", "swap", "rz", "rzz"]),
        InverseCancellation(gates_to_cancel=[CXGate()]),
    ]
)

In [17]:
tqc_rz = pm_rz.run(qc)
print_circuit_info(tqc_rz, 'Rz circuit')


swaps = []
for instruction in tqc_rz.data:
    if instruction.operation.name == 'swap':
        qubits_str = str(instruction.qubits)
        matches = re.findall('index=([0-9]+)', qubits_str)
        if len(matches) == 2:
            swaps.append((int(matches[0]), int(matches[1])))
        else:
            raise Exception('Did not find 2 swap indices')
for swap in swaps[::-1]:
    tqc_rz.swap(*swap)
tqc_rz.save_unitary()
res_rz = backend.run(tqc_rz).result()
data_rz = res_rz.results[0].data
unitary_rz = np.asarray(data_rz.unitary)

Max layers needed to apply swap decompose: 0
Gates we cannot directly implement: 0
[]
Transpiling accumulator
Rz circuit has 6 qubits
Rz circuit has 16 non-local gates and 14 non-local depth
Rz circuit contains ['cx', 'rz'] gates.


In [18]:
tqc_rz.draw(fold=-1)

unitary 
q_0 -> 0 ───────────────────────────────────────────■─────────────────■──────────────────────────────────────────■─────────────────■────────────────░────
                                                    │                 │                      ┌───┐┌───────────┐┌─┴─┐┌───────────┐┌─┴─┐┌───┐         ░    
q_1 -> 1 ──■────────────────────────────────────────┼─────────────────┼──────────────■───────┤ X ├┤ Rz(0.625) ├┤ X ├┤ Rz(0.625) ├┤ X ├┤ X ├─────────░────
           │  ┌───┐┌───────────┐┌───┐┌───────────┐┌─┴─┐┌───────────┐┌─┴─┐┌───┐┌───┐  │       └─┬─┘└───────────┘└───┘└───────────┘└───┘└─┬─┘         ░    
q_2 -> 2 ──┼──┤ X ├┤ Rz(0.625) ├┤ X ├┤ Rz(0.625) ├┤ X ├┤ Rz(0.625) ├┤ X ├┤ X ├┤ X ├──┼─────────┼────────────────────────────────────────┼───────────░────
         ┌─┴─┐└─┬─┘└───────────┘└─┬─┘└───────────┘└───┘└───────────┘└───┘└─┬─┘└─┬─┘┌─┴─┐┌───┐  │                                        │  ┌───┐    ░    
q_3 -> 3 ┤ X ├──■─────────────────┼────────────────────────────────────────┼────■──┤ X ├┤ X ├──■────────────────────────────────────────■──┤ X ├────░────
         ├───┤                    │                                        │  ┌───┐└───┘└─┬─┘                                              └─┬─┘    ░    
q_4 -> 4 ┤ X ├────────────────────■────────────────────────────────────────■──┤ X ├───────┼──────────────────────────────────────────────────┼──────░────
         └─┬─┘                                                                └─┬─┘       │                                                  │      ░    
q_5 -> 5 ──■────────────────────────────────────────────────────────────────────■─────────■──────────────────────────────────────────────────■──────░────
                                                                                                                                                    ░

In [19]:
np.nonzero(np.abs(unitary - unitary_rz) > 1e-5)

(array([], dtype=int64), array([], dtype=int64))